# 4.2 - Pemantauan Online Model Gen AI (Inference Table & Online Monitoring)

Notebook ini mempelajari cara membangun pipeline pemantauan waktu nyata (*online monitoring*) untuk model dan agen Gen AI di Databricks.

Ketika endpoint Model Serving diaktifkan dengan fitur **Inference Tables**, seluruh payload permintaan (*request*) dan tanggapan (*response*) dicatat secara otomatis ke dalam tabel Delta. Melalui notebook ini, kita memproses data payload tersebut menggunakan **Spark Structured Streaming** untuk mengekstrak percakapan serta menghitung metrik operasional dan keselamatan secara otomatis (seperti jumlah token dan skor toksisitas).

### Ringkasan Proses:
1. **Instalasi Pustaka & Setup**: Menginstal pustaka evaluasi & tokenisasi (`evaluate`, `tiktoken`, `transformers`, `torch`) serta menginisialisasi lingkungan lab dari `../Includes/Lab_Setup`.
2. **Inspeksi Inference Table**: Memeriksa skema dan isi tabel log inferensi mentah (`dphone_inference_payload`).
3. **Pembongkaran (*Unpacking*) Payload JSON**: Mendefinisikan logika untuk mengekstrak pesan input pengguna dan output model dari struktur JSON berlapis.
4. **Definisi Metrik Pemantauan**:
   - **Perhitungan Jumlah Token**: Menggunakan tokenizer `tiktoken` (`cl100k_base`) via Pandas UDF untuk mengestimasi konsumsi token dan biaya inferensi.
   - **Pengukuran Toksisitas (*Toxicity Measurement*)**: Menggunakan library Hugging Face `evaluate` via Pandas UDF untuk mendeteksi ujaran berbahaya atau melanggar etika.
5. **Pembuatan Tabel Hasil Pemantauan**: Membuat tabel Delta terkelola `dphone_processed_inferences` dengan Change Data Feed aktif.
6. **Eksekusi Pipeline Structured Streaming**: Membaca payload secara streaming, mengekstrak teks, menghitung metrik, dan menyimpan hasilnya secara otomatis dengan pemicu `availableNow=True`.


### Langkah 1: Instalasi Library Pendukung
Menginstal paket `evaluate` (pengukuran metrik kualitas teks), `tiktoken` (tokenisasi cepat OpenAI), `transformers`, dan `torch`, lalu merestart sesi Python agar perubahan library langsung aktif.


In [0]:
# Menginstal library pendukung pemantauan: evaluate (pengukuran metrik), tiktoken (penghitung token), transformers, dan torch
%pip install --quiet --upgrade evaluate tiktoken transformers torch
# Merestart kernel Python agar dependensi yang baru diinstal dapat langsung digunakan
dbutils.library.restartPython()

### Langkah 2: Setup Lingkungan Lab
Menjalankan notebook setup lab untuk menginisialisasi katalog, skema, volume cache, dan objek konfigurasi kursus.


In [0]:
%run ../Includes/Lab_Setup

### Langkah 3: Inspeksi Tabel Payload Inferensi (Inference Table)
Melihat struktur data mentah pada tabel `dphone_inference_payload`. Tabel ini otomatis diisi oleh Databricks Model Serving dan memuat kolom metadata seperti `request`, `response`, `status_code`, dan `timestamp`.


In [0]:
%sql
-- Menampilkan data rekaman inferensi mentah dari Databricks Model Serving Inference Table
SELECT * FROM dphone_inference_payload

### Langkah 4: Mendefinisikan Fungsi Pembongkaran Payload JSON (Unpack Requests)
Fungsi `unpack_requests` bertugas memfilter permintaan yang berhasil (`status_code == 200`), mengurai string JSON `request` dan `response` sesuai skema Spark SQL, serta mengambil pesan percakapan terakhir (`input` pengguna dan `output` model).


In [0]:
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.functions import col, element_at

def unpack_requests(requests_raw: DataFrame, 
                    input_json_path: str, 
                    input_schema: str, 
                    output_json_path: str, 
                    output_schema: str) -> DataFrame:
    """
    Membongkar kolom JSON request dan response pada Inference Table menjadi teks input dan output terstruktur.
    """
    # Hanya memproses rekaman yang menghasilkan status HTTP 200 (berhasil)
    requests_success = requests_raw.filter(col("status_code") == "200")

    # Mengurai string JSON menggunakan skema yang telah ditentukan
    requests_unpacked = (requests_success
        .withColumn("request", F.from_json(F.expr(f"request:{input_json_path}"), input_schema))
        .withColumn("response", F.from_json(F.expr(f"response:{output_json_path}"), output_schema))
    )

    # Mengambil pesan input terakhir dari pengguna dan pesan output terakhir dari model
    last_messages = (requests_unpacked.withColumn("input", F.element_at(F.col("request"), -1))
                                     .withColumn("output", F.element_at(F.col("response"), -1)[0])
                                     .drop("request", "response")
    )
    
    return last_messages

### Langkah 5: Menentukan Skema JSON Input dan Output
Menentukan path JSON dan skema data tipe Spark SQL yang sesuai dengan format pertukaran data API OpenAI / ResponsesAgent.


In [0]:
# Format input query: {"input": [{"content": "User question? ..."}]}
INPUT_PATH = "input[*].content"
INPUT_SCHEMA = "array<string>"

# Format output respons: {"output": [{"content": [{"text": "Model response ..."}]}]}
OUTPUT_PATH = "output[*].content[*].text"
OUPUT_SCHEMA = "array<array<string>>"

### Langkah 6: Pengujian Pembongkaran Payload pada Sampel Data
Menguji fungsi `unpack_requests` pada 10 baris sampel rekaman inferensi untuk memastikan teks pertanyaan dan jawaban terekstraksi dengan tepat.


In [0]:
# Menentukan nama lengkap tabel payload inferensi di Unity Catalog
inference_table_name = f"{course.catalog}.{course.schema}.dphone_inference_payload"

# Mengambil sampel 10 baris data inferensi sukses
sample_df = spark.table(inference_table_name).where('status_code == 200').limit(10)

# Menjalankan fungsi ekstraksi teks
unpacked_sample_df = unpack_requests(
    sample_df,
    INPUT_PATH,
    INPUT_SCHEMA,
    OUTPUT_PATH,
    OUPUT_SCHEMA
)

# Menampilkan DataFrame hasil ekstraksi
display(unpacked_sample_df)

### Langkah 7: Mendefinisikan Metrik Monitoring Menggunakan Pandas UDF
Mendefinisikan fungsi kalkulasi metrik terdistribusi yang dijalankan secara paralel di cluster Spark:
- `compute_num_tokens`: Menghitung jumlah token menggunakan tokenizer `tiktoken` (`cl100k_base`).
- `compute_toxicity`: Mengukur tingkat toksisitas teks menggunakan pustaka `evaluate` Hugging Face dengan penyimpanan cache lokal pada volume `{course.cache_volume}/huggingface`.


In [0]:
import os
import tiktoken, evaluate
import pandas as pd
from pyspark.sql.functions import pandas_udf

# Menentukan direktori cache untuk model Hugging Face di Unity Catalog Volume
cache_dir = f"{course.cache_volume}/huggingface"

# Pandas UDF untuk menghitung jumlah token teks secara efisien
@pandas_udf("int")
def compute_num_tokens(texts: pd.Series) -> pd.Series:
  encoding = tiktoken.get_encoding("cl100k_base")
  return pd.Series(map(len, encoding.encode_batch(texts)))
 
# Pandas UDF untuk mengukur tingkat toksisitas teks menggunakan model evaluate Hugging Face
@pandas_udf("double")
def compute_toxicity(texts: pd.Series) -> pd.Series:
  os.environ['HF_HOME'] = cache_dir
  import evaluate

  toxicity = evaluate.load("toxicity", module_type="measurement")
  return pd.Series(toxicity.compute(predictions=texts.fillna(""))["toxicity"]).where(texts.notna(), None)

### Langkah 8: Menghitung Metrik Pemantauan pada Kolom Input & Output
Fungsi `compute_metrics` menghitung jumlah token dan tingkat toksisitas untuk teks masukan pengguna (`input`) serta teks keluaran model (`output`), lalu menguji hasilnya pada data sampel.


In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col

def compute_metrics(requests_df: DataFrame, column_to_measure = ["input", "output"]) -> DataFrame:
  """Menambahkan kolom metrik token count dan toxicity untuk setiap kolom teks yang ditentukan."""
  for column_name in column_to_measure:
    requests_df = (
      requests_df.withColumn(f"token_count_{column_name}", compute_num_tokens(col(column_name)))
                 .withColumn(f"toxicity_{column_name}", compute_toxicity(col(column_name)))
    )
  return requests_df

# Menguji kalkulasi metrik pada sampel data
display(compute_metrics(unpacked_sample_df))

### Langkah 9: Membuat Tabel Delta Target Pemantauan
Membuat tabel Delta terkelola `dphone_processed_inferences` dengan Change Data Feed aktif (`delta.enableChangeDataFeed = true`) untuk menyimpan data inferensi yang sudah diperkaya dengan metrik pemantauan.


In [0]:
from delta.tables import DeltaTable

def create_processed_table_if_not_exists(table_name, requests_with_metrics):
    """Membuat tabel Delta target jika belum ada dan mengaktifkan Change Data Feed."""
    (
      DeltaTable.createIfNotExists(spark)
        .tableName(table_name)
        .addColumns(requests_with_metrics.schema)
        .property("delta.enableChangeDataFeed", "true")
        .execute()
    )

### Langkah 10: Menyiapkan Pipeline Structured Streaming Pemantauan
Menyiapkan pembacaan streaming (`readStream`) dari tabel inferensi, menerapkan pembersihan payload, menambahkan kalkulasi metrik online, dan memastikan tabel target telah siap menerima data.


In [0]:
# Menentukan lokasi penyimpanan checkpoint streaming pada Unity Catalog Volume
checkpoint_location = f"{course.cache_volume}/dphone_processed_inferences"

# Membaca tabel payload inferensi sebagai stream data terdistribusi
requests_raw_df = spark.readStream.table(inference_table_name)

# Mengekstrak teks dari payload JSON
requests_processed_df = unpack_requests(
    requests_raw_df,
    INPUT_PATH,
    INPUT_SCHEMA,
    OUTPUT_PATH,
    OUPUT_SCHEMA
)

# Menghapus kolom metadata yang tidak diperlukan untuk meminimalkan beban penyimpanan
requests_processed_df = requests_processed_df.drop("databricks_request_id", "request_date", "client_request_id", "status_code", "sampling_fraction", "logging_error_codes" , "requester")

# Menghitung metrik jumlah token dan skor toksisitas secara streaming
requests_with_metrics_df = compute_metrics(requests_processed_df)

# Menentukan nama tabel target di Unity Catalog dan memastikan tabel dibuat jika belum ada
processed_table_name = f"{course.catalog}.{course.schema}.dphone_processed_inferences"
create_processed_table_if_not_exists(processed_table_name, requests_with_metrics_df)

### Langkah 11: Menjalankan Eksekusi Streaming & Menampilkan Hasil Akhir
Menjalankan proses penulisan stream (`writeStream`) dengan pemicu `trigger(availableNow=True)` untuk memproses semua batch data inferensi yang tersedia, lalu menunggu hingga proses selesai (*awaitTermination*) dan menampilkan data tabel hasil pemantauan.


In [0]:
# Menuliskan data hasil pemantauan ke tabel target Delta Lake secara incremental
(requests_with_metrics_df.writeStream
                      .trigger(availableNow=True)
                      .outputMode("append")
                      .option("checkpointLocation", checkpoint_location)
                      .toTable(processed_table_name)
                      .awaitTermination())

# Menampilkan tabel Delta akhir yang memuat teks percakapan dan metrik pemantauan
display(spark.table(processed_table_name))